In [ ]:
import sys, pandas, numpy, sklearn, joblib
print(sys.version)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("sklearn:", sklearn.__version__)

In [ ]:
import ast
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

AUDIO_FEATURES = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence",
    "tempo", "duration_ms", "mode", "time_signature"
]

# Paths are relative to the repo root, so run this notebook from there.
RAW_DIR = Path("data/raw")   # tracks.csv and artists.csv from Kaggle go here
OUT_DIR = Path("data")       # the module notebooks read their inputs from here
OUT_DIR.mkdir(exist_ok=True)

tracks_path = RAW_DIR / "tracks.csv"
artists_path = RAW_DIR / "artists.csv"

for p in (tracks_path, artists_path):
    if not p.exists():
        raise FileNotFoundError(
            f"Missing {p}. Download tracks.csv and artists.csv from "
            "https://www.kaggle.com/datasets/yamaerenay/spotify-dataset-19212020-160k-tracks "
            f"and put them in {RAW_DIR}/"
        )

print("Raw data read from:", RAW_DIR.resolve())
print("Outputs will land at:", OUT_DIR.resolve())

In [ ]:
# Load raw CSVs
print("Loading tracks.csv ... (this takes ~10-20 seconds)")
tracks_raw = pd.read_csv(tracks_path)
print(f"  tracks shape: {tracks_raw.shape}")

print("Loading artists.csv ...")
artists_raw = pd.read_csv(artists_path)
print(f"  artists shape: {artists_raw.shape}")

# Expected counts (from Kaggle dataset spec)
print("\n--- Row-count check ---")
print(f"tracks:  {tracks_raw.shape[0]:>10,}   (expected 586,672)")
print(f"artists: {artists_raw.shape[0]:>10,}   (expected 1,162,095)")

# A first look at columns
print("\n--- tracks columns ---")
print(list(tracks_raw.columns))
print("\n--- artists columns ---")
print(list(artists_raw.columns))

In [ ]:
print("=" * 60)
print("TRACKS — dtypes and nulls")
print("=" * 60)
print(tracks_raw.dtypes)
print("\nNull counts per column:")
nulls_tracks = tracks_raw.isnull().sum()
print(nulls_tracks[nulls_tracks > 0] if nulls_tracks.sum() > 0 else "  (no nulls anywhere)")

print("\n" + "=" * 60)
print("ARTISTS — dtypes and nulls")
print("=" * 60)
print(artists_raw.dtypes)
print("\nNull counts per column:")
nulls_artists = artists_raw.isnull().sum()
print(nulls_artists[nulls_artists > 0] if nulls_artists.sum() > 0 else "  (no nulls anywhere)")

print("\n" + "=" * 60)
print("TRACKS — first 3 rows")
print("=" * 60)
print(tracks_raw.head(3).to_string())

print("\n" + "=" * 60)
print("ARTISTS — first 3 rows")
print("=" * 60)
print(artists_raw.head(3).to_string())

In [ ]:
# Start clean copies so raw stays untouched (useful for debugging)
tracks = tracks_raw.copy()
artists = artists_raw.copy()

print("Before cleaning:")
print(f"  tracks:  {len(tracks):>10,}")
print(f"  artists: {len(artists):>10,}")

# Drop rows with critical nulls
tracks = tracks.dropna(subset=["name"]).reset_index(drop=True)
artists = artists.dropna(subset=["name", "followers"]).reset_index(drop=True)

print("\nAfter dropping null names / followers:")
print(f"  tracks:  {len(tracks):>10,}   ({len(tracks_raw) - len(tracks):,} dropped)")
print(f"  artists: {len(artists):>10,}   ({len(artists_raw) - len(artists):,} dropped)")

# Confirm no nulls remain in critical columns
print("\nRemaining nulls in tracks:")
print(tracks.isnull().sum()[tracks.isnull().sum() > 0] if tracks.isnull().sum().sum() > 0 else "  (none)")
print("Remaining nulls in artists:")
print(artists.isnull().sum()[artists.isnull().sum() > 0] if artists.isnull().sum().sum() > 0 else "  (none)")

In [ ]:
# Look at what date formats actually appear in the data
sample_dates = tracks["release_date"].head(20).tolist()
print("Sample release_date values:")
for d in sample_dates:
    print(f"  {d!r}")

# Check for edge cases — any weird lengths?
date_lengths = tracks["release_date"].str.len().value_counts().sort_index()
print("\nDate string lengths (10=full date, 7=year-month, 4=year-only):")
print(date_lengths)

In [ ]:
# Strategy: take the first 4 chars — that's always the year, regardless of format
tracks["release_year"] = tracks["release_date"].str[:4].astype(int)
tracks["decade"] = (tracks["release_year"] // 10) * 10

# Sanity checks
print("\n--- Year range ---")
print(f"Min year: {tracks['release_year'].min()}")
print(f"Max year: {tracks['release_year'].max()}")

print("\n--- Rows per decade ---")
decade_counts = tracks["decade"].value_counts().sort_index()
print(decade_counts)

# Any weird / impossible years?
impossible = tracks[(tracks["release_year"] < 1900) | (tracks["release_year"] > 2025)]
print(f"\nRows with impossible years (<1900 or >2025): {len(impossible)}")
if len(impossible) > 0:
    print(impossible[["name", "release_date", "release_year"]].head())

In [ ]:
import time

# Also parse genres in artists table
print("Parsing stringified lists ... (this takes ~30-60s because of 586k rows)")

t0 = time.time()
tracks["artists"] = tracks["artists"].apply(ast.literal_eval)
tracks["id_artists"] = tracks["id_artists"].apply(ast.literal_eval)
print(f"  tracks list columns parsed in {time.time() - t0:.1f}s")

t0 = time.time()
artists["genres"] = artists["genres"].apply(ast.literal_eval)
print(f"  artists.genres parsed in {time.time() - t0:.1f}s")

# Extract the PRIMARY artist ID (first in the list) — this is what we'll join on
tracks["primary_artist_id"] = tracks["id_artists"].apply(lambda x: x[0] if len(x) > 0 else None)
tracks["primary_artist_name"] = tracks["artists"].apply(lambda x: x[0] if len(x) > 0 else None)

# Sanity check — verify types are now actual lists
print("\n--- Type sanity check ---")
print(f"tracks.artists[0]:            {tracks['artists'].iloc[0]!r}   (type: {type(tracks['artists'].iloc[0]).__name__})")
print(f"tracks.id_artists[0]:         {tracks['id_artists'].iloc[0]!r}   (type: {type(tracks['id_artists'].iloc[0]).__name__})")
print(f"tracks.primary_artist_id[0]:  {tracks['primary_artist_id'].iloc[0]!r}")
print(f"artists.genres[100]:          {artists['genres'].iloc[100]!r}   (type: {type(artists['genres'].iloc[100]).__name__})")

# How many tracks have no primary artist (shouldn't be many)
missing_primary = tracks["primary_artist_id"].isnull().sum()
print(f"\nTracks with missing primary artist: {missing_primary}")

# Distribution of number of artists per track (some songs have collaborations)
n_artists = tracks["artists"].apply(len)
print(f"\nNumber of artists per track (distribution):")
print(n_artists.value_counts().sort_index().head(10))

In [ ]:
# Build a small lookup: artist_id -> genres list, artist popularity, followers
artist_lookup = artists.set_index("id")[["genres", "popularity", "followers"]]
artist_lookup = artist_lookup.rename(columns={
    "genres": "artist_genres",
    "popularity": "artist_popularity",
    "followers": "artist_followers"
})

# Join onto tracks via primary_artist_id
print("Joining tracks with artist metadata ...")
tracks = tracks.merge(
    artist_lookup,
    how="left",
    left_on="primary_artist_id",
    right_index=True
)

print(f"tracks shape after join: {tracks.shape}")

# Sanity checks
n_matched = tracks["artist_genres"].notnull().sum()
n_unmatched = tracks["artist_genres"].isnull().sum()
print(f"\nTracks matched to an artist:   {n_matched:>10,}   ({100*n_matched/len(tracks):.2f}%)")
print(f"Tracks with NO artist match:   {n_unmatched:>10,}   ({100*n_unmatched/len(tracks):.2f}%)")

# For unmatched tracks, fill artist_genres with empty list so downstream code doesn't break
tracks["artist_genres"] = tracks["artist_genres"].apply(
    lambda x: x if isinstance(x, list) else []
)

# How many tracks have at least one genre tag?
n_with_genre = tracks["artist_genres"].apply(len).gt(0).sum()
print(f"\nTracks with >=1 genre tag:     {n_with_genre:>10,}   ({100*n_with_genre/len(tracks):.2f}%)")

# Top 15 most common genres across all tracks (flatten the list column)
print("\n--- Top 15 genres by track count ---")
all_genres = tracks["artist_genres"].explode().dropna()
print(all_genres.value_counts().head(15))

# Show a few enriched tracks
print("\n--- Sample of enriched tracks ---")
sample_cols = ["name", "primary_artist_name", "release_year", "artist_genres", "popularity"]
print(tracks.sample(5, random_state=SEED)[sample_cols].to_string())

In [ ]:
print(f"Before dedup: {len(tracks):,} tracks")

# A "song" is defined as (name, primary_artist_id).
# When duplicates exist, keep the row with the highest popularity — that's usually
# the canonical / most-streamed version and the most useful one for demos.
#
# Popularity is an integer from 0 to 100, so many duplicates tie (most early tracks sit at 0).
# Ties are broken by track `id`, which makes the order fully determined. Without a tie-breaker
# the winner depends on numpy's default quicksort, which is not stable and varies by CPU, so
# the cleaned set (and therefore the 100k sample) could differ between machines.
tracks_dedup = (
    tracks
    .sort_values(["popularity", "id"], ascending=[False, True])
    .drop_duplicates(subset=["name", "primary_artist_id"], keep="first")
    .sort_index()  # restore original order
    .reset_index(drop=True)
)

n_removed = len(tracks) - len(tracks_dedup)
print(f"After dedup:  {len(tracks_dedup):,} tracks   ({n_removed:,} removed, {100*n_removed/len(tracks):.1f}%)")

# Dedup checks: exactly one row per song, and it is that song's most popular version
song_max_pop = tracks.groupby(["name", "primary_artist_id"], dropna=False)["popularity"].max()
kept_pop = tracks_dedup.set_index(["name", "primary_artist_id"])["popularity"]
assert len(tracks_dedup) == len(song_max_pop), "Dedup left a different number of rows than there are songs"
assert kept_pop.index.is_unique, "Dedup left duplicate songs"
assert (kept_pop == song_max_pop.reindex(kept_pop.index)).all(), "Dedup did not keep the most popular version"
print("Dedup checks: OK (one row per song, highest-popularity version kept)")

# Show the songs that had the most copies in the raw data
# (grouped by artist ID, the same key the dedup uses)
dup_analysis = (
    tracks.groupby(["name", "primary_artist_id"], dropna=False)
    .agg(artist=("primary_artist_name", "first"), copies=("id", "size"))
    .sort_values("copies", ascending=False)
    .head(5)
    .reset_index(level="primary_artist_id", drop=True)
)
print("\n--- Songs that appeared most times in raw data ---")
print(dup_analysis)

# Overwrite tracks with the deduplicated version
tracks = tracks_dedup

# Post-dedup decade distribution
print("\n--- Post-dedup rows per decade ---")
print(tracks["decade"].value_counts().sort_index())

# Drop anything before 1920 (a single outlier row) for cleaner stratification
pre_1920 = tracks[tracks["decade"] < 1920]
print(f"\nPre-1920 rows: {len(pre_1920)} — will be dropped for cleaner stratification")
tracks = tracks[tracks["decade"] >= 1920].reset_index(drop=True)
print(f"Final tracks:  {len(tracks):,}")

In [ ]:
print("Fitting StandardScaler on the 12 audio features ...")
print(f"Features: {AUDIO_FEATURES}")

# Extract the feature matrix
X = tracks[AUDIO_FEATURES].values
print(f"Feature matrix shape: {X.shape}")

# Fit
scaler = StandardScaler()
scaler.fit(X)

# Show what the scaler learned — this becomes evidence in the report
print("\n--- Feature means (before scaling) ---")
for f, m in zip(AUDIO_FEATURES, scaler.mean_):
    print(f"  {f:20s}  mean = {m:>12.4f}")

print("\n--- Feature std deviations ---")
for f, s in zip(AUDIO_FEATURES, scaler.scale_):
    print(f"  {f:20s}  std  = {s:>12.4f}")

# Save it
scaler_path = OUT_DIR / "scaler.joblib"
joblib.dump(scaler, scaler_path)
print(f"\nScaler saved to: {scaler_path.resolve()}")

# Sanity check — reload it and verify identical
scaler_reloaded = joblib.load(scaler_path)
assert np.allclose(scaler.mean_, scaler_reloaded.mean_), "Reloaded scaler mismatch!"
assert np.allclose(scaler.scale_, scaler_reloaded.scale_), "Reloaded scaler mismatch!"
print("Reload verification: OK — scaler round-trips correctly.")

In [ ]:
# For each decade, compute mean and std of popularity, then z-score every song
print("Computing within-decade popularity z-scores ...")

decade_stats = tracks.groupby("decade")["popularity"].agg(["mean", "std"])
print("\n--- Popularity stats per decade ---")
print(decade_stats.round(2))

# Map back to the tracks dataframe and compute z-score
tracks["pop_decade_mean"] = tracks["decade"].map(decade_stats["mean"])
tracks["pop_decade_std"] = tracks["decade"].map(decade_stats["std"])
tracks["pop_z"] = (tracks["popularity"] - tracks["pop_decade_mean"]) / tracks["pop_decade_std"]

# Drop the helper columns — keep only pop_z
tracks = tracks.drop(columns=["pop_decade_mean", "pop_decade_std"])

# Sanity check the distribution
print("\n--- pop_z summary ---")
print(tracks["pop_z"].describe().round(3))

# Show the biggest hits *for their era* across a few decades
print("\n--- Top 3 hits (by pop_z) in each of a few decades ---")
for decade in [1960, 1980, 2000, 2020]:
    top = (
        tracks[tracks["decade"] == decade]
        .nlargest(3, "pop_z")[["name", "primary_artist_name", "popularity", "pop_z"]]
    )
    print(f"\n  {decade}s:")
    for _, r in top.iterrows():
        print(f"    pop_z={r['pop_z']:+.2f}  pop={r['popularity']:>3}  {r['primary_artist_name']} — {r['name']}")

In [ ]:
# Stratified sample: ~100k rows, with each decade keeping its share of the full dataset.

TARGET_SAMPLE_SIZE = 100_000

print(f"Building stratified sample of ~{TARGET_SAMPLE_SIZE:,} tracks ...")

# Compute sample fraction
sample_frac = TARGET_SAMPLE_SIZE / len(tracks)
print(f"Sample fraction: {sample_frac:.4f}")

# Take the same fraction from every decade. Proportional sampling matches the real-world
# distribution of music and avoids over-sampling the thin early decades. Every decade is
# sampled with the same seed, which is what reproduces the shared 100,001-row sample.
tracks_100k = pd.concat(
    [group.sample(frac=sample_frac, random_state=SEED)
     for _, group in tracks.groupby("decade")]
).reset_index(drop=True)

print(f"Sample size: {len(tracks_100k):,}")
print("\n--- Sample rows per decade ---")
print(tracks_100k["decade"].value_counts().sort_index())

# ---- Export the sample ----
# Only the 100k sample is saved. The full cleaned set (523,585 rows) is needed in memory
# above (scaler fit, pop_z stats, sampling frame), but no module notebook reads it from disk.
print("\n--- Exporting parquet file ---")

sample_path = OUT_DIR / "tracks_100k.parquet"

tracks_100k.to_parquet(sample_path, index=False)
print(f"  100k sample:    {sample_path.resolve()}   ({sample_path.stat().st_size / 1e6:.1f} MB)")

# Show final column list — this is the schema the team will consume
print("\n--- Final columns in the shared dataset ---")
for col in tracks_100k.columns:
    print(f"  {col:25s}  {str(tracks_100k[col].dtype):15s}")

In [ ]:
print("=" * 60)
print("VERIFICATION — reloading files from disk as a teammate would")
print("=" * 60)

# Reload from disk and compare against what this notebook just built in memory.
# Every check is an assert, so the final "PASSED" line only prints if all of them hold.
tracks_check = pd.read_parquet(OUT_DIR / "tracks_100k.parquet")
scaler_check = joblib.load(OUT_DIR / "scaler.joblib")

print(f"\nReloaded tracks_100k.parquet — shape: {tracks_check.shape}")
print(f"Reloaded scaler.joblib — type: {type(scaler_check).__name__}")

# Check 1 — the file on disk is exactly the sample built above
assert tracks_check.shape == tracks_100k.shape, "Row/column count differs from the in-memory sample"
assert list(tracks_check.columns) == list(tracks_100k.columns), "Columns differ from the in-memory sample"
assert (tracks_check["id"].values == tracks_100k["id"].values).all(), "Rows differ from the in-memory sample"
print("\nCheck 1 — parquet matches the in-memory sample: OK")

# Check 2 — list columns survive the parquet round-trip (they come back as numpy arrays)
for col in ["artists", "id_artists", "artist_genres"]:
    assert tracks_check[col].map(lambda x: isinstance(x, (list, np.ndarray))).all(), f"{col} is not list-like"
assert (tracks_check["artists"].map(len) == tracks_check["id_artists"].map(len)).all(), \
    "artists and id_artists have different lengths"
first_row = tracks_check.iloc[0]
print("Check 2 — list columns survive parquet: OK")
print(f"  artists:       {first_row['artists']!r}   (type: {type(first_row['artists']).__name__})")
print(f"  artist_genres: {first_row['artist_genres']!r}   (type: {type(first_row['artist_genres']).__name__})")

# Check 3 — the scaler is the full-dataset fit and transforms the sample cleanly
assert scaler_check.n_features_in_ == len(AUDIO_FEATURES), "Scaler expects a different number of features"
assert np.all(scaler_check.n_samples_seen_ == len(tracks)), "Scaler was not fit on the full cleaned dataset"
assert np.allclose(scaler_check.mean_, scaler.mean_) and np.allclose(scaler_check.scale_, scaler.scale_), \
    "Scaler on disk differs from the one fitted above"
X_scaled = scaler_check.transform(tracks_check[AUDIO_FEATURES].values)
assert X_scaled.shape == (len(tracks_check), len(AUDIO_FEATURES))
assert np.isfinite(X_scaled).all(), "Scaled features contain NaN or inf"
scaled_means, scaled_stds = X_scaled.mean(axis=0), X_scaled.std(axis=0)
# A proportional sample of ~19% of the data should sit close to mean 0 / std 1.
# (The std tolerance is wider because duration_ms has extreme outliers.)
assert np.all(np.abs(scaled_means) < 0.05), f"Scaled means drift from 0: {scaled_means.round(3)}"
assert np.all(np.abs(scaled_stds - 1) < 0.1), f"Scaled stds drift from 1: {scaled_stds.round(3)}"
print("Check 3 — scaler fit on full data, transforms sample cleanly: OK")
print(f"  scaled means: {scaled_means.round(3)}")
print(f"  scaled stds:  {scaled_stds.round(3)}")

# Check 4 — no nulls in the columns downstream code depends on
critical_cols = AUDIO_FEATURES + ["id", "name", "decade", "popularity", "pop_z", "primary_artist_id"]
null_counts = tracks_check[critical_cols].isnull().sum()
assert null_counts.sum() == 0, f"Nulls in critical columns:\n{null_counts[null_counts > 0]}"
print("Check 4 — no nulls in critical columns: OK")

# Check 5 — dedup and decade cleaning held in the sample
assert tracks_check["id"].is_unique, "Duplicate track ids in the sample"
assert not tracks_check.duplicated(subset=["name", "primary_artist_id"]).any(), "Duplicate songs in the sample"
assert (tracks_check["decade"] >= 1920).all(), "Pre-1920 rows in the sample"
assert (tracks_check["decade"] == (tracks_check["release_year"] // 10) * 10).all(), "decade and release_year disagree"
print("Check 5 — no duplicate songs, decades consistent: OK")

# Check 6 — stratification: each decade gets its proportional share (±1 for rounding),
# and every sampled row comes from the cleaned dataset
expected = tracks["decade"].value_counts() * sample_frac
actual = tracks_check["decade"].value_counts().reindex(expected.index, fill_value=0)
assert ((actual - expected).abs() <= 1).all(), \
    f"Decade counts off:\n{pd.DataFrame({'expected': expected.round(1), 'actual': actual})}"
assert set(tracks_check["id"]).issubset(set(tracks["id"])), "Sample contains rows not in the cleaned dataset"
print("Check 6 — decade proportions preserved: OK")

# Check 7 — pop_z is a true within-decade z-score
z_stats = tracks.groupby("decade")["pop_z"].agg(["mean", "std"])
assert np.allclose(z_stats["mean"], 0, atol=1e-9) and np.allclose(z_stats["std"], 1, atol=1e-9), \
    "pop_z is not standardized within each decade"
sample_z_mean, sample_z_std = tracks_check["pop_z"].mean(), tracks_check["pop_z"].std()
assert abs(sample_z_mean) < 0.05 and abs(sample_z_std - 1) < 0.05, "pop_z in the sample drifts from mean 0 / std 1"
print(f"Check 7 — pop_z standardized per decade: OK   (sample mean={sample_z_mean:.3f}, std={sample_z_std:.3f})")

print("\n" + "=" * 60)
print("ALL VERIFICATION CHECKS PASSED — files are ready to share")
print("=" * 60)